In [2]:
import os
from pathlib import Path
import re
import torch as th
from imitation.data import rollout
from imitation.data.types import Trajectory
from imitation.data import rollout
import numpy as np
from imitation.algorithms import bc
import gymnasium as gym
from imitation.data.types import Transitions
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from sdlarch_rl.utils.utils import get_last_index, GenericCNN
import gc
from IPython import get_ipython
import cv2
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecTransposeImage
from sdlarch_rl.utils.stf6_imitation import STF6Env
from stable_baselines3.common.atari_wrappers import WarpFrame
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3 import PPO
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.algorithms.adversarial.gail import GAIL

rng = np.random.default_rng(0)

demo_path = 'demos-sf6/'
train_path = 'imitation-sf6/'

ENT_WEIGHT= 1e-3 # 0 # 1e-4
BATCH_SIZE= 128 # 64 # 128 # 32 # 64 # 128
NUMBER_OF_EPOCH=20
EPOCH_PER_FILE=3 # 2
MINI_BATCH=64
L2=1e-5
learning_rate=2e-4
buffer_size = 6

NUM_ENV = 1
SAVE_DIR="./model-sf6"
TENSORBOARD="./tensorboard-sf6"
TOTAL_TIMESTEP_NUMB = 50_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
EVAL_FREQU=CHECK_FREQ_NUMB*2
MAX_STEPS= 14_000
LEARNING_RATING=2e-4
#EVALS=15
EVALS=30 # all

ENT_COEF = 0.00001
n_steps=2048
ppo_batch_size=128 * NUM_ENV

os.makedirs(train_path, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(TENSORBOARD, exist_ok=True)

def linear_schedule(initial_lr):
    def schedule(progress_remaining):
        return progress_remaining * initial_lr
    return schedule


def make_env():
    def _init():
        env = STF6Env()
        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=1)
        env = TimeLimit(env, max_steps=MAX_STEPS)
        return env
    return _init


env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=DummyVecEnv)
# env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=SubprocVecEnv)
env = VecFrameStack(env, 4, channels_order = "last")
env = VecTransposeImage(env)

obs = env.reset()

print("obs shape", obs[0].shape)

observation_space=env.observation_space
# observation_space = gym.spaces.Box(
#     low=0,
#     high=255,
#     shape=(4, 96, 96), # 4 frames 96x96
#     dtype=np.uint8,
# )
action_space=env.action_space

print("observation_space", observation_space.shape)
print("action_space", action_space.shape)

SAVE_DIR = Path(SAVE_DIR)
latest_model_path = get_latest_model(SAVE_DIR)

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    learner = PPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        learning_rate=linear_schedule(LEARNING_RATING),
        batch_size=ppo_batch_size,
    )
    
else:
    print("None finded, starting from zero.")
    learner = PPO("CnnPolicy", 
        env, 
        verbose=0, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=ppo_batch_size,
        tensorboard_log=TENSORBOARD, 
        learning_rate=linear_schedule(LEARNING_RATING)
    )


last_index = int(get_last_index(demo_path, "demos", "pt"))
last_index_imitation = int(get_last_index(train_path, "bc_policy", "zip"))

print("last_index: " + str(last_index))
print("Last training session saved: ", f"bc_policy{last_index_imitation}.zip")

# files_index = np.arange(last_index + 1)
files_index = np.arange(last_index + 1)

print("trainning files: ", files_index)


th.serialization.add_safe_globals([Trajectory])


reward_net = BasicRewardNet(
    observation_space=observation_space,
    action_space=action_space,
    normalize_input_layer=RunningNorm,
)

trainer = GAIL(
    demonstrations=None,
    demo_batch_size=1024,
    gen_replay_buffer_capacity=512,
    n_disc_updates_per_round=8,
    venv=env,
    gen_algo=learner,
    reward_net=reward_net,
    allow_variable_horizon=True,
)

def concat_transitions(list_of_transitions):
    return Transitions(
        obs=np.concatenate([t.obs for t in list_of_transitions]),
        acts=np.concatenate([t.acts for t in list_of_transitions]),
        next_obs=np.concatenate([t.next_obs for t in list_of_transitions]),
        dones=np.concatenate([t.dones for t in list_of_transitions]),
        infos=np.concatenate([t.infos for t in list_of_transitions]),
    )

def fix_action_format(acts):
    """
    Fix action shape
    """
    if isinstance(acts, np.ndarray):
        if acts.ndim == 3 and acts.shape[1] == 1:
            acts = acts.squeeze(1)
        
        # if acts.dtype == np.float32 or acts.dtype == np.float64:
        #     acts = np.round(acts).astype(np.int8)
    
    return acts

def fix_obs_to_hwc(obs: np.ndarray) -> np.ndarray:
    # (T, 1, H, W, C)
    if obs.ndim == 5 and obs.shape[1] == 1:
        obs = obs[:, 0]

    # (T, C, H, W, 1)
    if obs.ndim == 5 and obs.shape[-1] == 1:
        obs = obs.squeeze(-1)

    return obs

epoch_count = 0
for e in range(NUMBER_OF_EPOCH):
    np.random.shuffle(files_index)

    epoch_count += 1

    print(f"\n--------------- Epoch: {epoch_count} ------------------\n")

    print("files_index: ", files_index)

    buffer = []

    buffer_files = []

    for i in files_index:
        trajectories = th.load(demo_path + f"demos{i}.pt", weights_only=False)
        fixed_trajectories = []
            
        for traj in trajectories:
            obs = np.array(traj.obs)

            acts = fix_action_format(np.array(traj.acts, dtype=np.float32))

            obs = fix_obs_to_hwc(obs)
            
            fixed_trajectories.append(
                Trajectory(
                    obs=obs,
                    acts=acts,
                    infos=traj.infos,
                    terminal=traj.terminal
                )
            )

        ############### end for loop #######################

        
        np.random.shuffle(fixed_trajectories)
        
        transitions = rollout.flatten_trajectories(fixed_trajectories)

        buffer.append(transitions)
        buffer_files.append(i)

        if len(buffer) == buffer_size:
            merged = concat_transitions(buffer)

            print(f"Processing files: {buffer_files}")

            trainer.set_demonstrations(merged)
            # trainer.train(n_epochs=EPOCH_PER_FILE)
            trainer.train(2048)
            buffer.clear()
            buffer_files.clear()

        del transitions
        del fixed_trajectories
        del trajectories

trainer.policy.save(train_path + f"bc_policy{last_index_imitation + 1}.zip")

gc.collect()
trainer._demonstrations = None
trainer._demonstrations_tensor = None
del trainer

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

ModuleNotFoundError: No module named 'torch'